# Exp6 + Exp8 consensus ensemble inference

Load two checkpoints from two Kaggle datasets, run both detectors directly on the test images, and fuse their post-NMS detections.

Validated consensus settings (`mAP@0.5 = 0.853871`): match IoU `0.355`, unmatched-score penalty `0.220`, equal coordinate weights, and maximum `300` final detections per image.

In [ ]:
# Configuration
GITHUB_REPO_URL = "https://github.com/minh071289/YOLOv11-pt.git"
REPO_DIR = "/kaggle/working/YOLOv11-pt"

# Explicit paths are recommended because A and B use different NMS profiles.
# Leave both as None to auto-discover one checkpoint from each Kaggle dataset.
# Example explicit paths:
# CHECKPOINT_PATH_A = "/kaggle/input/my-exp8-weights/best.pth"
# CHECKPOINT_PATH_B = "/kaggle/input/my-exp6-weights/best.pth"
# TEST_IMAGE_DIR = "/kaggle/input/my-test-dataset/test/images"
CHECKPOINT_PATH_A = None
CHECKPOINT_PATH_B = None
TEST_IMAGE_DIR = None

# Model A is the Exp8 wider-neck checkpoint.
INPUT_SIZE_A = 640
CONFIDENCE_A = 0.0001
NMS_IOU_A = 0.50
MAX_DETECTIONS_A = 300

# Model B is the Exp6 baseline checkpoint.
INPUT_SIZE_B = 640
CONFIDENCE_B = 0.001
NMS_IOU_B = 0.55
MAX_DETECTIONS_B = 100

MATCH_IOU = 0.355
UNMATCHED_PENALTY = 0.220
ENSEMBLE_WEIGHT_A = 0.50
MAX_DETECTIONS = 300
BATCH_SIZE = 8

OUTPUT_JSON = "/kaggle/working/test_predictions.json"
OUTPUT_SUBMISSION = "/kaggle/working/submission.csv"
OUTPUT_SUMMARY = "/kaggle/working/inference_summary.json"

In [ ]:
# Clone the complete repository so all root scripts such as predict.py are available.
import os
import shutil
import subprocess
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run([
    "git", "clone", GITHUB_REPO_URL, str(repo_dir),
], check=True)

os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
print(f"Repository ready: {repo_dir}")

In [ ]:
# Locate two checkpoints and the test images from the attached Kaggle datasets.
from collections import Counter

IMAGE_EXTENSIONS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}
KAGGLE_INPUT = Path("/kaggle/input")

def checkpoint_rank(path):
    return (
        path.name.lower() == "best.pth",
        "best" in path.name.lower(),
        "exp8" in str(path).lower(),
        path.stat().st_size,
    )

def discover_checkpoints(root, count=2):
    candidates = list(root.rglob("*.pth"))
    by_dataset = {}
    for path in candidates:
        relative_parts = path.relative_to(root).parts
        dataset_name = relative_parts[0] if relative_parts else path.parent.name
        current = by_dataset.get(dataset_name)
        if current is None or checkpoint_rank(path) > checkpoint_rank(current):
            by_dataset[dataset_name] = path

    dataset_checkpoints = list(by_dataset.values())
    if len(dataset_checkpoints) < count:
        raise FileNotFoundError(
            f"Expected checkpoints from at least {count} Kaggle datasets; "
            f"found {len(dataset_checkpoints)}"
        )
    dataset_checkpoints.sort(key=checkpoint_rank, reverse=True)
    return dataset_checkpoints[:count]

def discover_image_dir(root):
    image_files = [
        path for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]
    if not image_files:
        raise FileNotFoundError("No test images found under /kaggle/input")
    counts = Counter(path.parent for path in image_files)
    return counts.most_common(1)[0][0]

if CHECKPOINT_PATH_A and CHECKPOINT_PATH_B:
    checkpoint_path_a = Path(CHECKPOINT_PATH_A)
    checkpoint_path_b = Path(CHECKPOINT_PATH_B)
elif CHECKPOINT_PATH_A or CHECKPOINT_PATH_B:
    raise ValueError("Set both CHECKPOINT_PATH_A and CHECKPOINT_PATH_B, or leave both as None")
else:
    checkpoint_path_a, checkpoint_path_b = discover_checkpoints(KAGGLE_INPUT)

test_image_dir = Path(TEST_IMAGE_DIR) if TEST_IMAGE_DIR else discover_image_dir(KAGGLE_INPUT)

if checkpoint_path_a.resolve() == checkpoint_path_b.resolve():
    raise ValueError("The two checkpoint paths must be different")
for checkpoint_path in (checkpoint_path_a, checkpoint_path_b):
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Checkpoint does not exist: {checkpoint_path}")
if not test_image_dir.is_dir():
    raise NotADirectoryError(f"Test image directory does not exist: {test_image_dir}")

image_paths = sorted(
    [path for path in test_image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS],
    key=lambda path: path.name,
)
if not image_paths:
    raise RuntimeError(f"No images directly inside: {test_image_dir}")

print(f"Checkpoint A: {checkpoint_path_a}")
print(f"Checkpoint B: {checkpoint_path_b}")
print(f"Test images: {test_image_dir}")
print(f"Number of images: {len(image_paths)}")

In [ ]:
# Load the exact architecture saved in each checkpoint.
import cv2
import torch
from tqdm.auto import tqdm

from predict import load_model
from utils import util
from utils.json_dataset import letterbox, scale_boxes_to_original

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Kaggle, select a GPU accelerator first.")

device = torch.device("cuda")
model_a, classes_a, checkpoint_input_size_a = load_model(checkpoint_path_a, device)
model_b, classes_b, checkpoint_input_size_b = load_model(checkpoint_path_b, device)
if classes_a != classes_b:
    raise ValueError(f"Checkpoint classes differ: {classes_a} vs {classes_b}")
classes = classes_a
if checkpoint_input_size_a != INPUT_SIZE_A:
    print(f"Checkpoint A input_size={checkpoint_input_size_a}; using {INPUT_SIZE_A}")
if checkpoint_input_size_b != INPUT_SIZE_B:
    print(f"Checkpoint B input_size={checkpoint_input_size_b}; using {INPUT_SIZE_B}")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Classes: {classes}")
print(
    f"Model A: input={INPUT_SIZE_A}, conf={CONFIDENCE_A}, "
    f"nms={NMS_IOU_A}, max_det={MAX_DETECTIONS_A}"
)
print(
    f"Model B: input={INPUT_SIZE_B}, conf={CONFIDENCE_B}, "
    f"nms={NMS_IOU_B}, max_det={MAX_DETECTIONS_B}"
)
print(
    f"Consensus: match_iou={MATCH_IOU}, penalty={UNMATCHED_PENALTY}, "
    f"weight_a={ENSEMBLE_WEIGHT_A}, max_det={MAX_DETECTIONS}"
)

In [ ]:
# Batched two-model CUDA inference followed by consensus fusion.
import csv
import json
import time
from pathlib import Path

def load_batch(paths, input_size):
    tensors = []
    metadata = []
    for image_path in paths:
        image = cv2.imread(str(image_path))
        if image is None:
            raise FileNotFoundError(f"Unable to read image: {image_path}")
        height, width = image.shape[:2]
        resized, ratio, pad = letterbox(image, input_size)
        tensor = torch.from_numpy(resized.transpose((2, 0, 1))[::-1].copy())
        tensors.append(tensor)
        metadata.append((image_path.name, ratio, pad, width, height))
    samples = torch.stack(tensors).to(device, non_blocking=True).float() / 255.0
    return samples, metadata

def format_prediction(detections, metadata):
    image_id, ratio, pad, width, height = metadata
    boxes = []
    if detections.shape[0]:
        detections = detections.detach().cpu()
        detections[:, :4] = scale_boxes_to_original(
            detections[:, :4], ratio, pad, width, height
        )
        for x1, y1, x2, y2, score, class_id in detections.tolist():
            if x2 <= x1 or y2 <= y1:
                continue
            boxes.append({
                "class": classes[int(class_id)],
                "confidence": round(float(score), 6),
                "bbox": [
                    round(float(x1), 3), round(float(y1), 3),
                    round(float(x2), 3), round(float(y2), 3),
                ],
            })
    boxes.sort(key=lambda item: item["confidence"], reverse=True)
    return {"image_id": image_id, "boxes": boxes}

def bbox_iou(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    intersection = max(x2 - x1, 0.0) * max(y2 - y1, 0.0)
    area_a = max(box_a[2] - box_a[0], 0.0) * max(box_a[3] - box_a[1], 0.0)
    area_b = max(box_b[2] - box_b[0], 0.0) * max(box_b[3] - box_b[1], 0.0)
    return intersection / max(area_a + area_b - intersection, 1e-9)

def fuse_detections(boxes_a, boxes_b):
    used_b = set()
    fused = []
    weight_b = 1.0 - ENSEMBLE_WEIGHT_A

    for box_a in boxes_a:
        best_index = -1
        best_iou = 0.0
        for index, box_b in enumerate(boxes_b):
            if index in used_b or box_a["class"] != box_b["class"]:
                continue
            overlap = bbox_iou(box_a["bbox"], box_b["bbox"])
            if overlap > best_iou:
                best_iou = overlap
                best_index = index

        if best_index >= 0 and best_iou >= MATCH_IOU:
            used_b.add(best_index)
            box_b = boxes_b[best_index]
            fused.append({
                "class": box_a["class"],
                "confidence": round(
                    (box_a["confidence"] * box_b["confidence"]) ** 0.5, 6
                ),
                "bbox": [
                    round(ENSEMBLE_WEIGHT_A * value_a + weight_b * value_b, 3)
                    for value_a, value_b in zip(box_a["bbox"], box_b["bbox"])
                ],
            })
        else:
            fused.append({
                **box_a,
                "confidence": round(box_a["confidence"] * UNMATCHED_PENALTY, 6),
            })

    for index, box_b in enumerate(boxes_b):
        if index not in used_b:
            fused.append({
                **box_b,
                "confidence": round(box_b["confidence"] * UNMATCHED_PENALTY, 6),
            })

    fused.sort(key=lambda item: item["confidence"], reverse=True)
    return fused[:MAX_DETECTIONS]

predictions = []
model_a_seconds = 0.0
model_b_seconds = 0.0
wall_start = time.perf_counter()

with torch.inference_mode():
    for start in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Detecting"):
        batch_paths = image_paths[start:start + BATCH_SIZE]
        samples_a, metadata_a = load_batch(batch_paths, INPUT_SIZE_A)
        if INPUT_SIZE_B == INPUT_SIZE_A:
            samples_b, metadata_b = samples_a, metadata_a
        else:
            samples_b, metadata_b = load_batch(batch_paths, INPUT_SIZE_B)

        torch.cuda.synchronize()
        infer_start = time.perf_counter()
        with torch.amp.autocast(device_type="cuda", enabled=True):
            outputs_a = model_a(samples_a)
        torch.cuda.synchronize()
        model_a_seconds += time.perf_counter() - infer_start

        infer_start = time.perf_counter()
        with torch.amp.autocast(device_type="cuda", enabled=True):
            outputs_b = model_b(samples_b)
        torch.cuda.synchronize()
        model_b_seconds += time.perf_counter() - infer_start
        outputs_a = outputs_a.float()
        outputs_b = outputs_b.float()

        for index in range(len(batch_paths)):
            detections_a = util.non_max_suppression(
                outputs_a[index:index + 1],
                confidence_threshold=CONFIDENCE_A,
                iou_threshold=NMS_IOU_A,
                max_detections=MAX_DETECTIONS_A,
            )[0]
            detections_b = util.non_max_suppression(
                outputs_b[index:index + 1],
                confidence_threshold=CONFIDENCE_B,
                iou_threshold=NMS_IOU_B,
                max_detections=MAX_DETECTIONS_B,
            )[0]
            prediction_a = format_prediction(detections_a, metadata_a[index])
            prediction_b = format_prediction(detections_b, metadata_b[index])
            if prediction_a["image_id"] != prediction_b["image_id"]:
                raise AssertionError("Model outputs were paired with different images")
            predictions.append({
                "image_id": prediction_a["image_id"],
                "boxes": fuse_detections(
                    prediction_a["boxes"], prediction_b["boxes"]
                ),
            })

wall_seconds = time.perf_counter() - wall_start
if len(predictions) != len(image_paths):
    raise AssertionError("Prediction count does not match image count")

Path(OUTPUT_JSON).write_text(
    json.dumps(predictions, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

# Competition submission format: exactly one row per image.
with Path(OUTPUT_SUBMISSION).open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["image_id", "bounding_boxes"])
    writer.writeheader()
    for prediction in predictions:
        submission_boxes = []
        for box in prediction["boxes"]:
            x1, y1, x2, y2 = box["bbox"]
            submission_boxes.append({
                "x_min": x1,
                "y_min": y1,
                "x_max": x2,
                "y_max": y2,
                "class": box["class"],
                "confidence": box["confidence"],
            })
        writer.writerow({
            "image_id": prediction["image_id"],
            "bounding_boxes": json.dumps(submission_boxes, ensure_ascii=False),
        })

summary = {
    "checkpoint_a": str(checkpoint_path_a),
    "checkpoint_b": str(checkpoint_path_b),
    "test_image_dir": str(test_image_dir),
    "num_images": len(image_paths),
    "num_predictions": sum(len(item["boxes"]) for item in predictions),
    "classes": classes,
    "model_a": {
        "input_size": INPUT_SIZE_A,
        "confidence": CONFIDENCE_A,
        "nms_iou": NMS_IOU_A,
        "max_detections": MAX_DETECTIONS_A,
    },
    "model_b": {
        "input_size": INPUT_SIZE_B,
        "confidence": CONFIDENCE_B,
        "nms_iou": NMS_IOU_B,
        "max_detections": MAX_DETECTIONS_B,
    },
    "consensus": {
        "match_iou": MATCH_IOU,
        "unmatched_penalty": UNMATCHED_PENALTY,
        "weight_a": ENSEMBLE_WEIGHT_A,
    },
    "max_detections_per_image": MAX_DETECTIONS,
    "ensemble": True,
    "tta": False,
    "amp": True,
    "wall_seconds": round(wall_seconds, 3),
    "wall_ms_per_image": round(1000 * wall_seconds / len(image_paths), 3),
    "model_a_seconds": round(model_a_seconds, 3),
    "model_b_seconds": round(model_b_seconds, 3),
    "total_model_ms_per_image": round(
        1000 * (model_a_seconds + model_b_seconds) / len(image_paths), 3
    ),
}
Path(OUTPUT_SUMMARY).write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print(json.dumps(summary, indent=2))
print(f"Saved: {OUTPUT_JSON}")
print(f"Saved submission: {OUTPUT_SUBMISSION}")
print(f"Saved: {OUTPUT_SUMMARY}")

In [ ]:
# Optional visual check. This does not alter the saved predictions.
import matplotlib.pyplot as plt

CLASS_COLORS = {
    "person": (255, 56, 56),
    "car": (56, 255, 56),
    "dog": (56, 56, 255),
    "cat": (255, 255, 56),
    "chair": (255, 56, 255),
}

preview_count = min(4, len(predictions))
figure, axes = plt.subplots(preview_count, 1, figsize=(14, 8 * preview_count))
if preview_count == 1:
    axes = [axes]

for axis, prediction in zip(axes, predictions[:preview_count]):
    image = cv2.imread(str(test_image_dir / prediction["image_id"]))
    # Draw only the top 30 boxes in the preview to keep it readable.
    for box in prediction["boxes"][:30]:
        x1, y1, x2, y2 = map(int, box["bbox"])
        color = CLASS_COLORS.get(box["class"], (255, 255, 255))
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        label = f'{box["class"]} {box["confidence"]:.3f}'
        cv2.putText(image, label, (x1, max(y1 - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.set_title(prediction["image_id"])
    axis.axis("off")

plt.tight_layout()
plt.show()